In [42]:
import os
import json
import pandas as pd

def parse_output_file(file_path):
    result = {}
    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            tokens = line.split(":")
            result[tokens[0].strip()] = tokens[1].strip()
    if 'Query throughput' in result:
        result['query_thput'] = float(result['Query throughput'].split()[0])
    if 'False positives' in result:
        result['fp'] = float(result['False positives'].split()[0])
    if 'FP queries' in result:
        result['fp'] = float(result['FP queries'].split()[0])
    if 'Time to query FP queries' in result:
        result['rent-time'] = float(result['Time to query FP queries'].split()[0])
    if 'Time to fix FP' in result:
        result['buy-time'] = float(result['Time to fix FP'].split()[0])
        result['breakeven'] = result['buy-time']/result['rent-time']
    return result





In [43]:
result_dir = "./results"
experiment_runs = []
for dir in os.listdir(result_dir):
    if dir == "_sources":
        continue
    experiment_run = {}
    experiment_run['id'] = str(dir)
    with open(os.path.join(result_dir, dir, 'config.json'), 'r', encoding='utf-8') as file:
        experiment_run['config'] = json.load(file)
    with open(os.path.join(result_dir, dir, 'run.json'), 'r', encoding='utf-8') as file:
        experiment_run['context'] = json.load(file)
    experiment_run['result'] = parse_output_file(os.path.join(result_dir, dir, 'output.txt'))
    experiment_runs.append(experiment_run)

df = pd.json_normalize(experiment_runs)
print(df.columns)
display(df)

Index(['id', 'config.distribution', 'config.filter', 'config.num_queries',
       'config.quotient_bits', 'config.remainder_bits', 'config.seed',
       'context.artifacts', 'context.command', 'context.experiment.base_dir',
       'context.experiment.dependencies', 'context.experiment.mainfile',
       'context.experiment.name', 'context.experiment.repositories',
       'context.experiment.sources', 'context.heartbeat', 'context.host.cpu',
       'context.host.gpus.driver_version', 'context.host.gpus.gpus',
       'context.host.hostname', 'context.host.os',
       'context.host.python_version', 'context.meta.command',
       'context.meta.config_updates.distribution',
       'context.meta.config_updates.filter',
       'context.meta.config_updates.num_queries',
       'context.meta.config_updates.quotient_bits',
       'context.meta.config_updates.remainder_bits',
       'context.meta.named_configs', 'context.meta.options.--beat-interval',
       'context.meta.options.--capture', 'cont

,id,config.distribution,config.filter,config.num_queries,config.quotient_bits,config.remainder_bits,config.seed,context.artifacts,context.command,context.experiment.base_dir,...,result.Successful attacks,result.Failed attacks,result.True positives,result.Time to query false positive queries,result.FP queries,result.Time to query FP queries,result.Time to fix FP,result.rent-time,result.buy-time,result.breakeven
0,54,u,nonAdaptive,100000000,26,10,516073594,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10,z,DAdaptive,10000000,24,8,771264382,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,77,z,DAdaptive,100000000,26,10,634989289,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,14,u,adaptive,10000000,24,8,874132907,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,49,z,adaptive,100000000,26,10,616124585,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,23,z,DAdaptive,100000000,26,10,561517275,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,66,z,nonAdaptive,100000000,26,10,428495950,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,72,u,DAdaptive,10000000,26,10,97263217,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,48,a,adaptive,100000000,26,10,462541717,"[output.txt, unif_q.csv, unif_i.csv]",run_experiment,/home/chesetti/Repos/skiAdaptiveQf/bench,...,35,87669,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Breakeven cost


In [48]:
filtered_df = df[df['config.distribution']=='m']
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter'])
        .agg({
            'result.fp': ['mean'],
            'result.breakeven': ['min', 'mean', 'max']
            }))

result.fp  \
                                                                                 mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter             
24                   8                     10000000           DAdaptive       70340.0   
26                   10                    100000000          DAdaptive      175966.0   

                                                                            result.breakeven  \
                                                                                         min   
config.quotient_bits config.remainder_bits config.num_queries config.filter                    
24                   8                     10000000           DAdaptive            28.219784   
26                   10                    100000000          DAdaptive            50.703476   

                                                                                        \
                                                                                  mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter              
24                   8                     10000000           DAdaptive      36.208308   
26                   10                    100000000          DAdaptive      82.049278   

                                                                                         
                                                                                    max  
config.quotient_bits config.remainder_bits config.num_queries config.filter              
24                   8                     10000000           DAdaptive       42.266603  
26                   10                    100000000          DAdaptive      128.379864

### Uniform query

In [49]:
filtered_df = df[df['config.distribution']=='u'].dropna(subset=['result.fp'])
print(df[df['config.distribution']=='u'][['id','result.fp']].dropna())
#display(filtered_df)
#df['result.False positives']
#display(filtered_df[['config.filter','config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'result.query_thput', 'result.False positives']])
#display(filtered_df[['config.filter','config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'result.query_thput', 'result.False positives']])
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.query_thput': ['mean', 'min', 'max']}))
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.fp': ['mean', 'min', 'max']}))

    id  result.fp
0   54    87840.0
3   14    35035.0
11  67    87407.0
13  87    88039.0
17  84    87918.0
19  57    87860.0
20   8    34682.0
25  75    88213.0
37  64    87454.0
42  37    88383.0
43  34    88110.0
58  11    34977.0
61  24    87813.0
63  78    88029.0
66   4    35237.0
67  44    87714.0
70  27    88160.0
75  71    35252.0
80  81    88008.0
87  47    87491.0


result.query_thput  \
                                                                                          mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                      
24                   8                     10000000           DAdaptive           1.024083e+07   
                                                              adaptive            1.486665e+06   
                                                              nonAdaptive         1.168996e+07   
26                   10                    100000000          DAdaptive           4.865539e+06   
                                                              adaptive            1.811743e+06   
                                                              nonAdaptive         5.116886e+06   

                                                                                           \
                                                                                      min   
config.quotient_bits config.remainder_bits config.num_queries config.filter                 
24                   8                     10000000           DAdaptive      9.521080e+06   
                                                              adaptive       1.486665e+06   
                                                              nonAdaptive    1.168996e+07   
26                   10                    100000000          DAdaptive      3.763357e+06   
                                                              adaptive       1.718785e+06   
                                                              nonAdaptive    4.595593e+06   

                                                                                           
                                                                                      max  
config.quotient_bits config.remainder_bits config.num_queries config.filter                
24                   8                     10000000           DAdaptive      1.108510e+07  
                                                              adaptive       1.486665e+06  
                                                              nonAdaptive    1.168996e+07  
26                   10                    100000000          DAdaptive      5.555160e+06  
                                                              adaptive       1.877817e+06  
                                                              nonAdaptive    5.659180e+06

result.fp  \
                                                                                 mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter             
24                   8                     10000000           DAdaptive       35057.0   
                                                              adaptive        35035.0   
                                                              nonAdaptive     34977.0   
26                   10                    100000000          DAdaptive       88041.4   
                                                              adaptive        87860.2   
                                                              nonAdaptive     87786.2   

                                                                                      \
                                                                                 min   
config.quotient_bits config.remainder_bits config.num_queries config.filter            
24                   8                     10000000           DAdaptive      34682.0   
                                                              adaptive       35035.0   
                                                              nonAdaptive    34977.0   
26                   10                    100000000          DAdaptive      87918.0   
                                                              adaptive       87407.0   
                                                              nonAdaptive    87454.0   

                                                                                      
                                                                                 max  
config.quotient_bits config.remainder_bits config.num_queries config.filter           
24                   8                     10000000           DAdaptive      35252.0  
                                                              adaptive       35035.0  
                                                              nonAdaptive    34977.0  
26                   10                    100000000          DAdaptive      88213.0  
                                                              adaptive       88383.0  
                                                              nonAdaptive    88110.0

### Zipfian Distribution


In [ ]:
filtered_df = df[df['config.distribution']=='z']
display(filtered_df[['id', 'config.filter', 'result.False positives', 'result.query_thput']])
#df['result.False positives']
display(filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.query_thput': ['mean', 'min', 'max'], 'result.fp': ['mean']}));


,id,config.filter,result.False positives,result.query_thput
1,10,DAdaptive,272,1.167692e+08
2,77,DAdaptive,279,8.285450e+07
4,49,adaptive,196,1.347456e+08
5,83,DAdaptive,284,8.315918e+07
8,80,DAdaptive,247,7.524998e+07
12,59,adaptive,191,1.312653e+08
23,56,nonAdaptive,4255,1.033855e+08
26,26,nonAdaptive,6449,1.107685e+08
29,43,DAdaptive,NaN,NaN
35,63,DAdaptive,NaN,NaN


result.query_thput  \
                                                                                          mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                      
24                   8                     10000000           DAdaptive           7.648069e+07   
                                                              adaptive            8.399268e+07   
                                                              nonAdaptive         9.131002e+07   
26                   10                    100000000          DAdaptive           1.144514e+08   
                                                              adaptive            1.269355e+08   
                                                              nonAdaptive         9.274354e+07   

                                                                                           \
                                                                                      min   
config.quotient_bits config.remainder_bits config.num_queries config.filter                 
24                   8                     10000000           DAdaptive      3.619215e+07   
                                                              adaptive       8.399268e+07   
                                                              nonAdaptive    9.131002e+07   
26                   10                    100000000          DAdaptive      7.524998e+07   
                                                              adaptive       9.670600e+07   
                                                              nonAdaptive    5.770770e+07   

                                                                                           
                                                                                      max  
config.quotient_bits config.remainder_bits config.num_queries config.filter                
24                   8                     10000000           DAdaptive      1.167692e+08  
                                                              adaptive       8.399268e+07  
                                                              nonAdaptive    9.131002e+07  
26                   10                    100000000          DAdaptive      1.698719e+08  
                                                              adaptive       1.363111e+08  
                                                              nonAdaptive    1.107685e+08

result.fp  \
                                                                                 mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter             
24                   8                     10000000           DAdaptive         264.0   
                                                              adaptive          198.0   
                                                              nonAdaptive      3391.0   
26                   10                    100000000          DAdaptive         269.8   
                                                              adaptive          189.0   
                                                              nonAdaptive      8589.4   

                                                                                     \
                                                                                min   
config.quotient_bits config.remainder_bits config.num_queries config.filter           
24                   8                     10000000           DAdaptive       256.0   
                                                              adaptive        198.0   
                                                              nonAdaptive    3391.0   
26                   10                    100000000          DAdaptive       247.0   
                                                              adaptive        176.0   
                                                              nonAdaptive    4255.0   

                                                                                      
                                                                                 max  
config.quotient_bits config.remainder_bits config.num_queries config.filter           
24                   8                     10000000           DAdaptive        272.0  
                                                              adaptive         198.0  
                                                              nonAdaptive     3391.0  
26                   10                    100000000          DAdaptive        284.0  
                                                              adaptive         203.0  
                                                              nonAdaptive    14952.0

### Adversarial test

In [53]:

filtered_df = df[df['config.distribution']=='a']
#display(filtered_df)
#df['result.False positives']
filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.query_thput': ['mean', 'min', 'max']})
filtered_df.groupby(['config.quotient_bits', 'config.remainder_bits', 'config.num_queries', 'config.filter']).agg({'result.query_thput': ['mean', 'min', 'max'], 'result.fp': ['mean']})

result.query_thput  \
                                                                                          mean   
config.quotient_bits config.remainder_bits config.num_queries config.filter                      
24                   8                     10000000           DAdaptive           1.623100e+06   
                                                              adaptive            1.613166e+06   
                                                              nonAdaptive         3.437145e+06   
26                   10                    100000000          DAdaptive           2.050832e+06   
                                                              adaptive            2.103931e+06   
                                                              nonAdaptive         9.729771e+05   

                                                                                           \
                                                                                      min   
config.quotient_bits config.remainder_bits config.num_queries config.filter                 
24                   8                     10000000           DAdaptive      1.618845e+06   
                                                              adaptive       1.613166e+06   
                                                              nonAdaptive    3.437145e+06   
26                   10                    100000000          DAdaptive      1.938000e+06   
                                                              adaptive       2.069227e+06   
                                                              nonAdaptive    8.539027e+05   

                                                                                           \
                                                                                      max   
config.quotient_bits config.remainder_bits config.num_queries config.filter                 
24                   8                     10000000           DAdaptive      1.627355e+06   
                                                              adaptive       1.613166e+06   
                                                              nonAdaptive    3.437145e+06   
26                   10                    100000000          DAdaptive      2.131646e+06   
                                                              adaptive       2.141130e+06   
                                                              nonAdaptive    1.159309e+06   

                                                                             result.fp  
                                                                                  mean  
config.quotient_bits config.remainder_bits config.num_queries config.filter             
24                   8                     10000000           DAdaptive        34919.5  
                                                              adaptive         35080.0  
                                                              nonAdaptive     533131.0  
26                   10                    100000000          DAdaptive        87677.0  
                                                              adaptive         87860.0  
                                                              nonAdaptive    5083356.6